# 06 — Gold RENAMU
## Transformación Silver → Gold (Refinamiento EAV)

A diferencia de otros pipelines Gold donde se aplica el Unpivot, aquí la Capa Silver ya entrega
el modelo **EAV (Entidad-Atributo-Valor)** completo. El objetivo de este cuaderno es refinarlo:

- Filtrar respuestas vacías residuales
- Normalizar y homogeneizar nombres de preguntas entre años
- Agregar métricas de cobertura por municipalidad y por año

**Fuente:** `data/silver/renamu/` (EAV: UBIGEO | ANO_RENAMU | MODULO | PREGUNTA | RESPUESTA)  
**Destino:** `data/gold/stg_renamu/` (Parquet Gold particionado por ANO_RENAMU)

In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F

from src.paths import SILVER, GOLD, str_path, ensure_dirs
from src.spark_utils import get_spark
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

In [ ]:
SILVER_PATH = str_path(SILVER["renamu"])
GOLD_PATH   = str_path(GOLD["stg_renamu"])

print(f"Silver: {SILVER_PATH}")
print(f"Gold  : {GOLD_PATH}")
ensure_dirs()

In [ ]:
spark = get_spark(app_name="MEF_Gold_RENAMU", memory="4g")

## PARTE 1: Lectura Silver EAV

In [ ]:
silver_dir = SILVER["renamu"]
if not silver_dir.exists() or not any(silver_dir.rglob("*.parquet")):
    raise FileNotFoundError("Sin datos Silver RENAMU. Ejecutar: 03_silver_renamu.ipynb")

df = spark.read.parquet(SILVER_PATH)

n_raw = df.count()
print(f"Filas EAV en Silver : {n_raw:,}")
print(f"Columnas            : {df.columns}")
print(f"Anos disponibles    :")
df.groupBy("ANO_RENAMU").count().orderBy("ANO_RENAMU").show(20)

## PARTE 2: Refinamiento y Normalización

- Eliminamos respuestas vacías o literalmente nulas que pudieron haber pasado el filtro de Silver.
- Normalizamos el nombre de las preguntas (trim + upper) para facilitar la comparación entre años.

In [ ]:
gold_df = (
    df
    # Filtrar respuestas vacías residuales
    .filter(
        F.col("RESPUESTA").isNotNull() &
        (F.trim(F.col("RESPUESTA")) != "") &
        (~F.trim(F.col("RESPUESTA")).isin(["None", "none", "null", "NULL", "N/A", "nan"]))
    )
    # Normalizar nombres de pregunta para comparación consistente entre años
    .withColumn("PREGUNTA", F.upper(F.trim(F.col("PREGUNTA"))))
    # Asegurar que UBIGEO es siempre 6 dígitos
    .withColumn("UBIGEO", F.lpad(F.col("UBIGEO").cast("string"), 6, "0"))
)

n_gold = gold_df.count()
print(f"Filas tras refinamiento Gold: {n_gold:,}")
print(f"Eliminadas en refinamiento  : {n_raw - n_gold:,}")

## PARTE 3: Escritura a Gold

In [ ]:
control = ControlManager(pipeline_name="gold_renamu")
execution = control.start(input_parameters={"silver_rows": n_raw})

start_time = time.time()
try:
    (
        # coalesce(1): consolida ~54 MB en 1 archivo; sin particionado anual
    gold_df.coalesce(1).write
        .mode("overwrite")
        .format("parquet")
        .save(GOLD_PATH)
    )
    elapsed = time.time() - start_time
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={"silver_rows": n_raw, "gold_rows": n_gold, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"Gold RENAMU completado en {elapsed:.1f}s")
    print(f"   {n_raw:,} filas Silver -> {n_gold:,} filas Gold")

except Exception as e:
    control.log_error("GoldRenamuError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

In [ ]:
# ── Verificación Final ────────────────────────────────────────────────────────
df_result = spark.read.parquet(GOLD_PATH)
print(f"Verificacion Gold RENAMU:")
print(f"  Total filas  : {df_result.count():,}")
print(f"  Columnas     : {df_result.columns}")
print(f"\n  Top 10 preguntas mas frecuentes:")
df_result.groupBy("PREGUNTA").count().orderBy("count", ascending=False).show(10, truncate=False)
print(f"\n  Cobertura por ano:")
df_result.groupBy("ANO_RENAMU").agg(
    F.countDistinct("UBIGEO").alias("municipalidades"),
    F.count("RESPUESTA").alias("total_respuestas")
).orderBy("ANO_RENAMU").show(20)